# Figure 4.4

In [1]:
import sys, os, io, contextlib
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from Ranking_exp import Ranking_exp

# ------------------------- parameters -------------------------
n = 9
p = 6
q = 4
k_s = 1.0
k_d = 0.0
delta_s = k_s - k_d

k_o_vals   = np.arange(0.40, 0.499, 0.001)
c_var_vals = np.arange(1e-4, 4, 0.0025)
c_vals     = 1.0 / (c_var_vals * delta_s)

test_critical_pair  = [q - 1, p+1]      # (q-1, p): test generalization margin
train_critical_pair = [q, q + 1]      # (q, q+1): training margin

# ------------------------- sweep -------------------------
breaking_reg_test  = []
breaking_reg_train = []

for k_o in tqdm(k_o_vals, desc='k_o sweep'):

    # -- find smallest c_var where TEST margin flips negative --
    test_flag = False
    for i, cr in enumerate(c_vals):
        with contextlib.redirect_stdout(io.StringIO()):
            sim = Ranking_exp(n=n, k_o=k_o, k_s=k_s, k_d=k_d,
                              p=p, q=q, c_reg=cr)
        if sim.f_j_k(test_critical_pair[0], test_critical_pair[1]) < 0:
            breaking_reg_test.append(c_var_vals[i])
            test_flag = True
            break
    if not test_flag:
        breaking_reg_test.append(0)

    # -- find smallest c_var where TRAIN margin flips negative --
    train_flag = False
    for i, cr in enumerate(c_vals):
        with contextlib.redirect_stdout(io.StringIO()):
            sim = Ranking_exp(n=n, k_o=k_o, k_s=k_s, k_d=k_d,
                              p=p, q=q, c_reg=cr)
        if sim.f_j_k(train_critical_pair[0], train_critical_pair[1]) < 0:
            breaking_reg_train.append(c_var_vals[i])
            train_flag = True
            break
    if not train_flag:
        breaking_reg_train.append(0)

# ------------------------- plot -------------------------
train   = np.array(breaking_reg_train)
test    = np.array(breaking_reg_test)
overlap = np.minimum(train, test)

with plt.rc_context({'figure.dpi': 200}):
    fig, ax = plt.subplots(figsize=(3.5, 2.5))

    ax.plot(k_o_vals, train, lw=1.0, color='#1f77b4',
            label='Training boundary (memorization)')
    ax.plot(k_o_vals, test,  lw=1.0, color='#ff7f0e',
            label='Test boundary (generalization)')

    ax.fill_between(k_o_vals, overlap, train, alpha=0.12, color='#1f77b4')
    ax.fill_between(k_o_vals, overlap, test,  alpha=0.12, color='#ff7f0e')
    ax.fill_between(k_o_vals, 0, overlap, alpha=0.20, color='#2ca02c',
                    label='Correct on both')

    ax.set_xlabel(r'$k_o$', fontsize=6, labelpad=2)
    ax.set_ylabel(r'$c^{-1}$', fontsize=6, rotation=0, labelpad=2)
    ax.set_title(
        f'Phase diagram: representational geometry vs. regularization\n'
        f'$n={n}$, $p={p}$, $q={q}$',
        fontsize=6, pad=4
    )
    ax.legend(fontsize=5, framealpha=0.9, edgecolor='gray', loc='upper right')
    ax.tick_params(axis='both', length=2, pad=1, labelsize=6)

    ax.set_xlim(0.40, 0.50)
    ax.set_ylim(bottom=0)
    ax.grid(False)

    plt.tight_layout(pad=0.5)
    plt.show()
    fig.savefig("figure_4e.pdf", bbox_inches="tight")

k_o sweep:   0%|          | 0/99 [00:00<?, ?it/s]

KeyboardInterrupt: 